# Baseline profiling: Kaggle Airbnb test dataset

Ноутбук загружает `data/kaggle-airbnb/test.csv`, строит baseline-профиль с помощью `profiling.baseline_profiler.Profiler` и выводит основные части результата.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
import rich
from rich import inspect

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from drift_guardian.core.profiling.baseline_profiler import Profiler

REFERRENCE_DATAPATH = PROJECT_ROOT / 'data' / 'kaggle-airbnb' / 'train.parquet'
CURRENT_DATAPATH = PROJECT_ROOT / 'data' / 'kaggle-airbnb' / 'test.parquet'

print(f'Reference datapath: {REFERRENCE_DATAPATH}')
print(f'Current datapath: {CURRENT_DATAPATH}')


Reference datapath: /Users/yulia/Projects/drift_guardian/data/kaggle-airbnb/train.parquet
Current datapath: /Users/yulia/Projects/drift_guardian/data/kaggle-airbnb/test.parquet


## Загрузка и первичный обзор данных

In [ ]:
reference_df = pd.read_parquet(REFERRENCE_DATAPATH, engine="pyarrow")

print(f'Размер датасета: {reference_df.shape[0]:,} строк × {reference_df.shape[1]} колонок')
reference_df.dtypes

Размер датасета: 36,671 строк × 15 колонок


name                    str
_id                   int64
host_name               str
location_cluster        str
location                str
lat                 float64
lon                 float64
type_house              str
sum                   int64
min_days              int64
amt_reviews           int64
last_dt                 str
avg_reviews         float64
total_host            int64
target                int64
dtype: object

## Настройка профилирования

`_id` — технический идентификатор, `name` и `host_name` — свободный высококардинальный текст, а `last_dt` — дата. Они исключены из baseline drift-признаков. Текущая версия профайлера не поддерживает временные признаки.

In [ ]:
NUM_FEATURES = [
    'lat',
    'lon',
    'sum',
    'min_days',
    'amt_reviews',
    'avg_reviews',
    'total_host',
]
CAT_FEATURES = [
    'location_cluster',
    'location',
    'type_house',
]
TARGET = 'target'
EXCLUDED_FEATURES = ['_id', 'name', 'host_name', 'last_dt']
WINDOW_SIZE = 1000

pd.Series({
    'numeric_features': NUM_FEATURES,
    'categorical_features': CAT_FEATURES,
    'excluded_features': EXCLUDED_FEATURES,
    'window_size': WINDOW_SIZE,
})

numeric_features        [lat, lon, sum, min_days, amt_reviews, avg_rev...
categorical_features             [location_cluster, location, type_house]
excluded_features                         [_id, name, host_name, last_dt]
window_size                                                          1000
dtype: object

## Построение baseline-профиля

Класс `Profiler` строит **baseline-профиль эталонного датасета**. Эталонным обычно выступает обучающая выборка или исторические данные за период, когда распределение признаков считалось нормальным.

Профиль содержит компактное статистическое описание эталонных данных. В дальнейшем новые батчи или потоковые окна сравниваются с этим описанием для обнаружения data drift.

<pre style="text-align:center; font-size:13px">
Эталонный датасет
↓
Profiler
↓
Baseline-профиль
↓
Сравнение с текущими данными
↓
Drift-метрики и алерты
</pre>

Профайлер обрабатывает только столбцы, перечисленные в списках:
- `num_features`
- `cat_features`
- `target`

Остальные столбцы игнорируются.

#### **`num_features`**: числовые признаки
Для каждого числового признака профайлер рассчитывает:
- `n`: количество непустых значений;
- `missing_rate`: долю пропусков;
- `mean`: среднее значение;
- `std`: стандартное отклонение;
- `min`, `max`: минимум и максимум;
- `quantiles`: квантили `p01`, `p05`, `p10`, `p25`, `p50`, `p75`, `p90`, `p95`, `p99`;
- `decile_bins={edges, frequencies, proportions}`: границы децильных бакетов, количество и долю значений в каждом бакете.
- `low_cardinality`

Кроме агрегатов, профайлер сохраняет выборку исходных числовых значений — `reference sample`. Она может использоваться для KS-теста, Wasserstein distance и других статистических сравнений.

#### Категориальные признаки
Для каждого категориального признака профайлер рассчитывает:
- `n`: количество непустых значений;
- `missing_rate`: долю пропусков;
- `categories`: список известных категорий;
- `proportions`: частоту каждой категории;
- ``: долю каждой категории;
- ``: список редких категорий;
- ``: суммарную долю редких категорий.
Категория считается редкой, если она встретилась меньше `merge_threshold` раз. По умолчанию:
`merge_threshold = 5`.


'feature': 'type_house',
    'type': 'categorical',
    'n': np.int64(36671),
    'missing_rate': np.float64(0.0),
    'categories': словарь {категория: количество строк}
    'proportions': {
        'Entire home/apt': 0.5184750893076273,
        'Private room': 0.4576640942434076,
        'Shared room': 0.023860816448965122
    },
    'is_complete_category_list': True,
    'merge_info': {
        'merge_threshold': 5,
        'other_bucket': {'categories': [], 'is_catch_all_for_unseen': True, 'proportion': np.float64(0.0)}
    },
    'churn_baseline': 'reference'


Профайлер записывает редкие категории в `merge_info`, чтобы при последующем анализе их можно было объединить в общий бакет `OTHER`. Само объединение на этапе профилирования пока не выполняется.


Функция `__init__` класса `Profiler` принимает следующие параметры:

<pre style="background-color: #383d47; border: 1px solid #3d414a; padding: 10px; border-radius: 4px; color: #d8dee9; font-family: monospace; font-size:13px; line-height:18px">
ref_data: pd.DataFrame,
window_size: int,
num_features: list | None = None,
cat_features: list | None = None,
target: str | None = None,
merge_threshold: int = 5,
low_cardinality_threshold: int = 15,
sample_dtype="float32",
random_state: int | None = None
</pre>

**`num_features`**: список числовых признаков для профилирования.

`cat_features`: список категориальных признаков для профилирования.
Категориальные признаки получают следующие характеристики:
    'feature': 'type_house',
    'type': 'categorical',
    'n': np.int64(36671),
    'missing_rate': np.float64(0.0),
    'categories': словарь {категория: }
    'proportions': {
        'Entire home/apt': 0.5184750893076273,
        'Private room': 0.4576640942434076,
        'Shared room': 0.023860816448965122
    },
    'is_complete_category_list': True,
    'merge_info': {
        'merge_threshold': 5,
        'other_bucket': {'categories': [], 'is_catch_all_for_unseen': True, 'proportion': np.float64(0.0)}
    },
    'churn_baseline': 'reference'

- `target`: необязательное имя целевой переменной. Target профилируется отдельно и возвращается в target_ref вместе с исходными значениями. Если target не указан в `num_features` или `cat_features`, профайлер автоматически определяет его как числовой либо категориальный. При профилировании target не добавляется в обычные num_ref или cat_ref, а записывается отдельно. Автоматически определённый boolean target будет признан числовым.
- `merge_threshold`: минимальная частота категории, при которой она не считается редкой. Категории, встретившиеся меньше указанного количества раз, записываются в merge_info как кандидаты для общего бакета OTHER. По умолчанию значение равно 5. На этапе профилирования категории только помечаются, но фактически не объединяются.
- `low_cardinality_threshold`: максимальное количество уникальных значений, при котором числовой признак считается низкокардинальным. Такой признак получает одновременно числовые и категориальные характеристики. Децильные бакеты для него не сохраняются. По умолчанию значение равно 15.
- `sample_dtype`: NumPy-тип данных, в который преобразуется reference sample числового признака. Разрешены только floating-типы, например float32 или float64. По умолчанию используется float32, чтобы уменьшить расход памяти.

In [ ]:
profiler = Profiler(
    ref_data=reference_df,
    window_size=WINDOW_SIZE,
    num_features=NUM_FEATURES,
    cat_features=CAT_FEATURES,
    target=TARGET,
    random_state=42,
)

cat_ref, num_ref, samples, target_ref = profiler.profile_ref_data()

print('Профиль успешно построен')
print(f'Числовых профилей: {len(num_ref)}')
print(f'Категориальных профилей: {len(cat_ref)}')
print(f'Reference samples: {len(samples)}')
print(f'Target задан: {bool(target_ref)}')

Профиль успешно построен
Числовых профилей: 7
Категориальных профилей: 3
Reference samples: 7
Target задан: False


In [19]:
print([type(x) for x in profiler.profile_ref_data()])

[<class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>]


In [57]:
# inspect(num_ref, methods=True)

In [10]:
print([*cat_ref])
print([*num_ref])


['location_cluster', 'location', 'type_house']
['lat', 'lon', 'sum', 'min_days', 'amt_reviews', 'avg_reviews', 'total_host']


In [11]:
sample_num_feature = 'sum'
rich.print(num_ref[sample_num_feature])

{
    'type': 'numeric',
    'n': np.int64(36671),
    'missing_rate': np.float64(0.0),
    'mean': np.float64(152.14229227454936),
    'std': np.float64(239.10797261205252),
    'max': np.int64(10000),
    'min': np.int64(0),
    'quantiles': {
        'p01': 30.0,
        'p05': 40.0,
        'p10': 49.0,
        'p25': 69.0,
        'p50': 106.0,
        'p75': 175.0,
        'p90': 269.0,
        'p95': 350.0,
        'p99': 760.0
    },
    'decile_bins': {
        'edges': [-inf, 49.0, 60.0, 75.0, 90.0, 106.0, 130.0, 155.0, 199.0, 269.0, inf],
        'frequencies': array([3504, 3033, 3967, 3742, 4089, 3473, 3790, 3445, 3939, 3689]),
        'proportions': array([0.09555234, 0.08270841, 0.10817812, 0.10204249, 0.111505  ,
       0.09470699, 0.10335142, 0.09394344, 0.10741458, 0.1005972 ])
    },
    'low_cardinality': False
}

In [14]:
sample_cat_feature = 'type_house'
rich.print(cat_ref[sample_cat_feature])

{
    'feature': 'type_house',
    'type': 'categorical',
    'n': np.int64(36671),
    'missing_rate': np.float64(0.0),
    'categories': {'Entire home/apt': 19013, 'Private room': 16783, 'Shared room': 875},
    'proportions': {
        'Entire home/apt': 0.5184750893076273,
        'Private room': 0.4576640942434076,
        'Shared room': 0.023860816448965122
    },
    'is_complete_category_list': True,
    'merge_info': {
        'merge_threshold': 5,
        'other_bucket': {'categories': [], 'is_catch_all_for_unseen': True, 'proportion': np.float64(0.0)}
    },
    'churn_baseline': 'reference'
}

## Результат: числовые признаки

In [6]:
numeric_summary = pd.DataFrame.from_dict(num_ref, orient='index')[
    ['n', 'missing_rate', 'mean', 'std', 'min', 'max', 'low_cardinality']
]
numeric_summary.index.name = 'feature'
display(numeric_summary)

,n,missing_rate,mean,std,min,max,low_cardinality
feature,,,,,,,
lat,12224,0.000000,40.728555,0.054105,40.50641,40.91306,False
lon,12224,0.000000,-73.952806,0.046020,-74.23914,-73.71690,False
sum,12224,0.000000,154.455825,243.267298,0.00000,9999.00000,False
min_days,12224,0.000000,6.960324,17.055669,1.00000,365.00000,False
amt_reviews,12224,0.000000,22.796875,44.086754,0.00000,607.00000,False
avg_reviews,9674,0.208606,1.396199,1.785811,0.01000,58.50000,False
total_host,12224,0.000000,7.432837,34.001000,1.00000,327.00000,False


In [7]:
numeric_quantiles = pd.DataFrame({
    feature: profile['quantiles']
    for feature, profile in num_ref.items()
}).T
numeric_quantiles.index.name = 'feature'
display(numeric_quantiles)

,p01,p05,p10,p25,p50,p75,p90,p95,p99
feature,,,,,,,,,
lat,40.596275,40.647266,40.667819,40.690117,40.722765,40.762543,40.804345,40.825046,40.863490
lon,-74.027591,-74.004194,-73.996820,-73.983362,-73.956125,-73.937255,-73.908536,-73.869663,-73.773873
sum,30.000000,40.000000,49.000000,70.000000,106.000000,178.000000,268.000000,369.000000,800.000000
min_days,1.000000,1.000000,1.000000,1.000000,3.000000,5.000000,29.000000,30.000000,45.000000
amt_reviews,0.000000,0.000000,0.000000,1.000000,5.000000,23.000000,69.000000,111.000000,210.770000
avg_reviews,0.020000,0.040000,0.060000,0.190000,0.710000,2.040000,3.750000,4.743500,7.240000
total_host,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,5.000000,15.000000,232.000000


## Результат: категориальные признаки

In [8]:
categorical_summary = pd.DataFrame.from_dict({
    feature: {
        'n': profile['n'],
        'missing_rate': profile['missing_rate'],
        'categories': len(profile['categories']),
        'rare_categories': len(
            profile['merge_info']['other_bucket']['categories']
        ),
        'other_proportion': (
            profile['merge_info']['other_bucket']['proportion']
        ),
    }
    for feature, profile in cat_ref.items()
}, orient='index')
categorical_summary.index.name = 'feature'
display(categorical_summary)

,n,missing_rate,categories,rare_categories,other_proportion
feature,,,,,
location_cluster,12224,0.0,5,0,0.000000
location,12224,0.0,204,66,0.011535
type_house,12224,0.0,3,0,0.000000


In [9]:
for feature, profile in cat_ref.items():
    top_categories = (
        pd.Series(profile['proportions'], name='proportion')
        .sort_values(ascending=False)
        .head(10)
        .to_frame()
    )
    print(f'{feature}: top-10 категорий')
    display(top_categories)

location_cluster: top-10 категорий
location: top-10 категорий
type_house: top-10 категорий


,proportion
Manhattan,0.447317
Brooklyn,0.411895
Queens,0.112647
Bronx,0.020861
Staten Island,0.007281


,proportion
Bedford-Stuyvesant,0.078452
Williamsburg,0.078207
Harlem,0.053665
Bushwick,0.049084
East Village,0.039267
Upper East Side,0.039185
Upper West Side,0.038694
Hell's Kitchen,0.038449
Midtown,0.032641
Crown Heights,0.031577


,proportion
Entire home/apt,0.523233
Private room,0.453452
Shared room,0.023315


## Результат: reference samples

Профайлер формирует отсортированные выборки числовых признаков размером `max(5000, 10 × window_size)`, но не больше числа непустых значений в исходных данных.

In [10]:
sample_summary = pd.DataFrame.from_dict({
    feature: {
        'sample_size': len(sample),
        'dtype': str(sample.dtype),
        'min': sample.min(),
        'max': sample.max(),
    }
    for feature, sample in samples.items()
}, orient='index')
sample_summary.index.name = 'feature'
display(sample_summary)

,sample_size,dtype,min,max
feature,,,,
lat,10000,float32,40.506409,40.913059
lon,10000,float32,-74.239143,-73.716904
sum,10000,float32,0.000000,9999.000000
min_days,10000,float32,1.000000,365.000000
amt_reviews,10000,float32,0.000000,597.000000
avg_reviews,9674,float32,0.010000,58.500000
total_host,10000,float32,1.000000,327.000000
